# Training Metrics Review
This notebook groups the metrics saved by `MetricsStorage` and `validate` so we can quickly spot obvious issues.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebooks.review_utils import (
    PROJECT_ROOT,
    list_metric_files,
    load_metrics,
    metrics_to_frame,
    plot_metric_groups,
    plot_run_comparison,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 120)
PROJECT_ROOT

In [ ]:
metric_files = list_metric_files()
display(metric_files.head(20))

In [ ]:
selected_metrics_path = None
if not metric_files.empty:
    selected_metrics_path = PROJECT_ROOT / metric_files.iloc[0]['metrics_file']
selected_metrics_path

In [ ]:
if selected_metrics_path is not None:
    metrics = load_metrics(selected_metrics_path)
    metric_frame = metrics_to_frame(metrics)
    display(metric_frame.head(20))
    display(metric_frame.groupby('metric')['value'].agg(['count', 'min', 'max', 'mean']).sort_index())

In [ ]:
if selected_metrics_path is not None:
    plot_metric_groups(metrics)

In [ ]:
if selected_metrics_path is not None:
    epoch_series = metrics.get('epoch', [])
    for metric_name in ['reg_loss_train', 'reg_loss_val', 'cls_loss_train', 'cls_loss_val', 'reg_error_val', 'accuracy']:
        if metric_name in metrics:
            print(metric_name, 'points=', len(metrics[metric_name]), 'epochs=', epoch_series[:len(metrics[metric_name])])

In [ ]:
comparison_candidates = [PROJECT_ROOT / path for path in metric_files.head(3)['metrics_file'].tolist()]
if comparison_candidates:
    plot_run_comparison(comparison_candidates, metric_name='reg_error_val')

## Notes
- Validation metrics are appended only on validation epochs, so many training series are denser than validation series.
- `outliers_count_val` is stored as a total count over the whole validation pass, while most others are averaged by batch.
- This notebook is aimed at sanity-checking trends first, not at final reporting.